In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
url = "http://127.0.0.1:8000/sample.html"

In [ ]:
res = requests.get(url) # 페이지 요청
res.text[:500]

In [ ]:
soup = BeautifulSoup(res.text, 'html.parser')
soup.title

In [ ]:
ul_tag = soup.find("ul", class_="product-list")
li_tag = ul_tag.find_all("li")
li_tag

In [ ]:
# find()로 상품 목록을 감싸고 있는 ul 태그를 찾습니다.
product_list = soup.find("ul", class_="product-list")

# find_all()로 모든 상품을 찾습니다.
product_items = product_list.find_all("li", class_="product-card")

for product in product_items:
    product_name = product.find("h3", class_="product-name").get_text(strip=True)
    price = product.find("span", class_="price").get_text(strip=True)

    print({"상품": product_name, "가격": price})

In [ ]:
import pandas as pd

# 상품 카드에서 상품명과 가격을 추출합니다.
product_list = soup.find("ul", class_="product-list")
product_items = product_list.find_all("li", class_="product-card")

product_data = []
for product in product_items:
    product_data.append({
        "상품": product.find("h3", class_="product-name").get_text(strip=True),
        "가격": product.find("span", class_="price").get_text(strip=True),
    })

products_df = pd.DataFrame(product_data)

# 가격에서 통화 기호와 쉼표를 제거해 숫자로 변환합니다.
products_df["가격(원)"] = (
    products_df["가격"]
    .str.replace(r"[^0-9]", "", regex=True)
    .astype(int)
)

products_df[["상품", "가격"]]

In [ ]:
res = requests.get("https://stock.naver.com/market/marketindex")
res.status_code, res.text[:800]

In [ ]:
soup = BeautifulSoup(res.text,"html.parser")
soup.find("div")

In [ ]:
url = "https://finance.naver.com/marketindex/exchangeList.naver"
m_index = requests.get(url)
soup = BeautifulSoup(m_index.content, 'html.parser')


table = soup.find('table', class_='tbl_exchange')
usd_row = table.find('tbody').find('tr')
us_price = usd_row.find(class_='sale')
print(us_price.text)


# 야후 파이낸스 환율 API 호출하기

In [ ]:
import requests
import yfinance as yf
from bs4 import BeautifulSoup

currency = "USD"
tickers = {"USD": "KRW=X", "JPY": "JPYKRW=X", "EUR": "EURKRW=X"}
ticker = tickers.get(currency)

if not ticker:
    print(f"{currency}는 지원하지 않는 토오하입니다.")

price = yf.Ticker(ticker).info.get("regularMarketPrice")
price

In [ ]:
yf.Ticker(ticker).info

In [ ]:
class Book:
    def __init__(self, rank, title, author, price):
        self.rank = rank
        self.title = title
        self.author = author
        self.price = price


    def __str__(self):
        return f"{self.rank}, {self.title}, {self.author}, {self.price}"
   
    def to_dict(self):
        return {'rank':self.rank,
                'title':self.title,
                'author':self.author,
                'price':self.price}
   
    def to_list(self):
        return [self.rank,
                self.title,
                self.author,
                self.price]


In [ ]:
# https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001 의 국내도서 종합베스트 순위를 가져온다.
import builtins
import pandas as pd
from IPython.display import display
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

page_no = 3
rank = 0
books = []

for page in range(1, page_no + 1):
    yes_url = f"https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001&pageNumber={page}"
    res = requests.get(yes_url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    best_list_el = soup.select("#yesBestList div.item_info")

    for item in best_list_el:
        rank += 1
        title_el = item.select_one("div.info_name > a")
        author_el = item.select_one("span.info_auth")
        price_el = item.select_one("div.info_price strong .yes_b")

        if not all((title_el, author_el, price_el)):
            continue

        books.append(Book(
            rank=rank,
            title=title_el.get_text(strip=True),
            author=author_el.get_text(" ", strip=True),
            price=price_el.get_text(strip=True),
        ))

# Book 객체를 딕셔너리로 변환한 뒤 DataFrame으로 만듭니다.
books_df = pd.DataFrame([book.to_dict() for book in books])
books_df["가격(원)"] = (
    books_df["price"]
    .str.replace(r"[^0-9]", "", regex=True)
    .astype(int)
)

# pandas 표에서 보기 좋은 한글 컬럼명으로 변경합니다.
books_df = books_df.rename(columns={
    "rank": "순위",
    "title": "제목",
    "author": "저자",
    "price": "가격",
})

books_df[["순위", "제목", "저자", "가격(원)"]]

In [ ]:
import sqlite3

# books.db 데이터베이스에 연결합니다. 파일이 없으면 새로 생성됩니다.
conn = sqlite3.connect("my_database.db")

create_books_table = """
CREATE TABLE IF NOT EXISTS books (
    book_rank INTEGER NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price INTEGER NOT NULL
);
"""

conn.execute(create_books_table)
conn.commit()

# 기존 테이블에 id 컬럼이 있으면 삭제합니다.
columns = [row[1] for row in conn.execute("PRAGMA table_info(books)")]
if "id" in columns:
    conn.execute("ALTER TABLE books RENAME TO books_old")
    conn.execute("""
        CREATE TABLE books (
            book_rank INTEGER NOT NULL,
            title TEXT NOT NULL,
            author TEXT NOT NULL,
            price INTEGER NOT NULL
        )
    """)
    conn.execute("""
        INSERT INTO books (book_rank, title, author, price)
        SELECT book_rank, title, author, price
        FROM books_old
    """)
    conn.execute("DROP TABLE books_old")
    conn.commit()

print("books 테이블이 생성되었습니다.")

In [ ]:
insert_sql = """
INSERT INTO books (book_rank, title, author, price)
VALUES (?, ?, ?, ?)
"""

# DataFrame의 컬럼 순서를 INSERT 컬럼 순서에 맞춥니다.
book_rows = books_df[["순위", "제목", "저자", "가격(원)"]]

# 모든 도서를 한 번에 삽입합니다.
conn.executemany(
    insert_sql,
    book_rows.itertuples(index=False, name=None),
)
conn.commit()

print(f"{len(book_rows)}개의 도서가 books 테이블에 저장되었습니다.")


In [ ]:
import builtins

# 도서를 저장한 books.db에 연결합니다.
conn = sqlite3.connect("my_database.db")

select_sql = """
SELECT *
FROM books
WHERE book_rank < 10
ORDER BY book_rank
"""

cursor = conn.execute(select_sql)
rows = cursor.fetchall()

if rows:
    for book_rank, title, author, price in rows:
        builtins.print(f"{book_rank} | {title} | {author} | {price:,}")
else:
    builtins.print("조회 결과가 없습니다.")